# 图文数据处理（合并优化版）

把原「图文数据处理-标准版」和「图文数据处理-标准版 - 到货订单」两个 notebook 合并成一份，
**一次运行同时产出两个汇总文件**：

| 输出文件 | 口径 | 来源 |
| --- | --- | --- |
| `直播汇总.xlsx` | 全量订单 | 原「标准版」 |
| `直播汇总-到货.xlsx` | 只统计已签收订单（到货） | 原「到货订单」版 |

## 两个版本的差异 & 合并方式

| 差异点 | 标准版 | 到货订单版 | 合并处理 |
| --- | --- | --- | --- |
| 销售金额 | 商品数量×商品单价 − 商家实际承担优惠 + 运费 | 商品数量×商品单价 + 运费 | 两列都算：`销售金额` / `到货销售金额`，各口径各取各的 |
| 包裹状态映射 | 已签收/派送中/待取件 → 已完成1 | 仅「已签收」→ 已完成1 | 先保留 `包裹状态_原始`，到货口径 = 原始状态「已签收」 |
| 统计范围 | 全部订单 | 仅到货订单 | 同一管线，最后按口径切分 |
| 输出 / 检查表 | 直播汇总.xlsx、检查.xlsx | 直播汇总-到货.xlsx、检查.xlsx（覆盖） | 输出两个文件；检查表分别写 检查.xlsx / 检查-到货.xlsx |

## 本次做的优化

- **逐行 `for` 循环向量化**：原代码有 5 处 `for i in range(len(df3))` + `df3.loc[i, ...]`（订单状态覆写、优惠券、站内订单、订单数归零、状态指标），全部改为整列布尔掩码，几万行数据从"分钟级"降到"毫秒级"。
- **修复链式 `inplace`**：`df2["包裹状态"].replace(..., inplace=True)`、`df3['售前退款单量'].fillna(0, inplace=True)` 等写法在 pandas 3 的 Copy-on-Write 下**是空操作**，改为显式赋值。
- **去掉已废弃写法**：`ExcelWriter.save()` 在 pandas 2.0 已移除，改用 `with` 上下文管理器。
- **公共逻辑复用**：两个版本的指标计算、透视、比率格式完全一致，抽成 `add_metrics()` / `build_tables()`，只保留 4 处真正的口径差异。
- **兼容性**：不用 Python 3.10+ 语法（`X | None`）、不用 pandas 2.0+ 才有的参数、路径用字符串，Python 3.7 / 旧版 pandas 可直接跑。
- **清理**：删除注释掉的死代码、空单元格、未使用的中间列。

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__}")

# ---- 文件路径（用字符串，兼容 Python 3.7+ / 旧版 pandas） ------------------
ORDER_CSV = "./1.csv"                  # 订单明细
PACKAGE_XLSX = "./2.xlsx"              # 包裹状态（运单）
SKU_XLSX = "./商品编码.xlsx"            # 商品编码 → 产品名称
CONSUME_XLSX = "./店铺消耗.xlsx"        # 广告消耗

SUMMARY_FILE = "直播汇总.xlsx"                 # 全量汇总
SUMMARY_ARRIVAL_FILE = "直播汇总-到货.xlsx"     # 到货汇总
LACK_FILE = "缺码表.xlsx"                      # 缺码表
CHECK_FILE = "检查.xlsx"                       # 中间结果（全量）
CHECK_ARRIVAL_FILE = "检查-到货.xlsx"           # 中间结果（到货）
WRITE_CHECK = True                             # 不需要中间结果时改成 False

# ---- 包裹状态映射 & 到货口径 ---------------------------------------------
PACKAGE_TO_DONE = {"已签收": "已完成1", "派送中": "已完成1", "待取件": "已完成1"}
ARRIVAL_RAW_STATUS = "已签收"

# ---- 售后状态分组 ---------------------------------------------------------
DONE_AFTER_SALE = {"换货成功", "-", "售后关闭", "补寄/换货成功", "补寄成功"}
AFTER_SALE_IN_PROGRESS_EXCLUDE = {"换货成功", "-", "售后关闭", "退款成功"}

# ---- 透视 / 输出列定义（与原脚本完全一致） --------------------------------
PIVOT_VALUES = [
    "销售金额", "订单数", "在线支付已完成", "售后中", "退回成功", "已发货",
    "复购", "广告", "总支付单量", "售前退款单量", "备货单量", "揽件前退款单量",
    "揽件订单数", "发货时长（小时）", "揽件时长（小时）", "站内订单", "优惠券单量",
]
RESULT_COLUMNS = [
    "订单提交时间", "产品名称", "订单数", "销售金额", "售后中", "在线支付已完成",
    "已发货", "退回成功", "复购", "广告", "总支付单量", "售前退款单量", "备货单量",
    "揽件前退款单量", "揽件订单数", "发货时长（小时）", "揽件时长（小时）",
    "站内订单", "优惠券单量",
]
FINAL_COLUMNS = [
    "日期", "产品名称", "广告单量", "总发出单量", "真实发货率", "总支付单量",
    "售前退款率", "揽件前退款率", "在线支付已完成", "已发货", "售后中", "退回成功",
    "签收率", "退回率", "复购率", "消耗", "发货时长", "揽件时长", "优惠订单占比",
    "站内订单占比", "对标支付签收率", "对标揽件单量签收率", "对标揽件单量退回率",
    "对标揽件单量复购率", "平均客单", "客单", "复购", "备货单量", "售前退款单量",
    "揽件前退款单量", "发货时长（小时）", "揽件时长（小时）", "站内订单",
    "优惠券单量", "揽件订单数",
]
ADVERT_COLUMNS = [
    "日期", "产品名称", "广告单量", "总发出单量", "真实发货率", "总支付单量",
    "售前退款率", "揽件前退款率", "在线支付已完成", "已发货", "售后中", "退回成功",
    "签收率", "退回率", "消耗", "发货时长", "揽件时长", "优惠订单占比",
    "站内订单占比", "对标支付签收率", "对标揽件单量签收率", "对标揽件单量退回率",
    "平均客单", "客单", "备货单量", "售前退款单量", "揽件前退款单量",
    "发货时长（小时）", "揽件时长（小时）", "站内订单", "优惠券单量", "揽件订单数",
]
RENAME_COLUMNS = {
    "转化数": "广告单量",
    "订单数": "总发出单量",
    "订单提交时间": "日期",
    "销售金额": "客单",
}
RATE_COLUMNS = [
    "真实发货率", "签收率", "退回率", "复购率", "对标支付签收率", "站内订单占比",
    "优惠订单占比", "对标揽件单量签收率", "对标揽件单量退回率",
    "对标揽件单量复购率", "售前退款率", "揽件前退款率",
]

Python 3.7.0 | pandas 0.23.4


## 1. 读取订单

In [2]:
df = pd.read_csv(ORDER_CSV, encoding="gbk")

# 兼容带制表符前缀的时间
if "\t" in str(df.loc[0, "订单提交时间"]):
    df["订单提交时间"] = df["订单提交时间"].str[1:]

## 2. 基础字段清洗（商家编码 / 时间 / 两种销售金额口径）

In [3]:
df["商家编码"] = df["商家编码"].str[1:]
df["订单提交时间"] = pd.to_datetime(df["订单提交时间"], format="%Y-%m-%d %H:%M:%S")
df["订单提交时间"] = df["订单提交时间"].dt.strftime("%Y-%m-%d")

# 全量版口径：商品数量×商品单价 − 商家实际承担优惠 + 运费
df["销售金额"] = (df["商品数量"] * df["商品单价"]) - df["商家实际承担优惠金额"] + df["运费"]
# 到货版口径：商品数量×商品单价 + 运费
df["到货销售金额"] = (df["商品数量"] * df["商品单价"]) + df["运费"]

## 3. 关联商品编码，导出缺码表

In [4]:
sku = pd.read_excel(SKU_XLSX).rename(columns={"商品名": "产品名称", "商品编码": "商家编码"})
df = df.merge(sku[["商家编码", "产品名称"]], on="商家编码", how="left")

lack_code = df.loc[df["产品名称"].isnull(), "商家编码"].value_counts()
lack_code.to_excel(LACK_FILE)
print(f"缺码 {len(lack_code)} 个，已导出 {LACK_FILE}")

缺码 0 个，已导出 缺码表.xlsx


## 4. 剔除直播渠道订单

In [5]:
df = df[df["广告渠道"] != "直播"]
df = df.reset_index(drop=True)
print(df["售后状态"].value_counts(dropna=False).head())

-       6775
退款成功    1250
待收退货      59
售后关闭      30
待退货       25
Name: 售后状态, dtype: int64


## 5. 合并包裹状态（保留原始状态，供到货口径使用）

In [6]:
df2 = pd.read_excel(PACKAGE_XLSX, converters={"订单编号": str})
df2 = df2.rename(columns={"订单编号": "主订单编号"})
df["主订单编号"] = df["主订单编号"].astype(str)
df2["主订单编号"] = df2["主订单编号"].astype(str)
df2 = df2[["主订单编号", "包裹状态", "揽件时间"]]

# 记住原始包裹状态：到货口径 = 原始状态为「已签收」
df2["包裹状态_原始"] = df2["包裹状态"]
df2["包裹状态"] = df2["包裹状态"].replace(PACKAGE_TO_DONE)
df2 = df2.drop_duplicates(subset="主订单编号", keep="first")

if "\t" in str(df2.loc[0, "主订单编号"]):
    df2["主订单编号"] = df2["主订单编号"].str[:19]

df3 = df.merge(df2, on="主订单编号", how="left")
df3["主订单编号"] = df3["主订单编号"].astype(str)
df3["子订单编号"] = df3["子订单编号"].astype(str)

# 包裹已完成 + 售后正常 → 订单状态记为已完成（原逐行 for 循环的等价写法）
df3.loc[
    df3["包裹状态"].eq("已完成1") & df3["售后状态"].isin({"-", "售后关闭", "换货成功"}),
    "订单状态",
] = "已完成"

print(df3["包裹状态"].value_counts(dropna=False))

已完成1    5589
NaN     1174
运输中      752
已发货      503
已退回      105
已取消       25
Name: 包裹状态, dtype: int64


## 6. 导出检查表（全量）

In [7]:
# 去掉合并过程中新增的辅助列，保持与原「检查.xlsx」列结构一致
_check_cols = ["包裹状态_原始", "到货销售金额"]
if WRITE_CHECK:
    df3.drop(columns=_check_cols).to_excel(CHECK_FILE, index=False)
    print(f"已导出 {CHECK_FILE}")

已导出 检查.xlsx


## 7. 切分到货订单（已签收）

In [8]:
df3_arrival = df3[df3["包裹状态_原始"] == ARRIVAL_RAW_STATUS].copy()
print(f"全量订单行：{len(df3):,} | 到货（已签收）行：{len(df3_arrival):,}")

if WRITE_CHECK:
    # 到货口径的检查表：销售金额换成到货公式，列结构与原到货版一致
    check_arrival = df3_arrival.drop(columns=_check_cols).copy()
    check_arrival["销售金额"] = df3_arrival["到货销售金额"]
    check_arrival.to_excel(CHECK_ARRIVAL_FILE, index=False)
    print(f"已导出 {CHECK_ARRIVAL_FILE}")

全量订单行：8,148 | 到货（已签收）行：5,128
已导出 检查-到货.xlsx


## 8. 指标计算（两个口径共用）

原代码 5 处逐行 `for` 循环全部改为整列布尔掩码。

In [9]:
def add_metrics(df_in, sales_col="销售金额"):
    """补齐单量、状态、时长、复购等指标。sales_col 指定销售金额取值列。"""
    out = df_in.copy()

    # ① 订单数先按商品数量计（后续重算）
    out["订单数"] = out["商品数量"]

    # ② 剔除未支付订单：订单状态=已关闭 且 售后状态=-
    out = out[~((out["订单状态"] == "已关闭") & (out["售后状态"] == "-"))].reset_index(drop=True)

    # ③ 支付 / 退款 / 备货单量
    out["总支付单量"] = out["订单数"]
    out["售前退款单量"] = np.where(
        out["包裹状态"].isna() & out["售后状态"].eq("退款成功"), out["订单数"], 0
    )
    out["揽件前退款单量"] = np.where(
        out["揽件时间"].isna() & out["售后状态"].eq("退款成功"), out["订单数"], 0
    )
    out["备货单量"] = np.where(
        out["包裹状态"].isna() & out["订单状态"].eq("待发货"), out["订单数"], 0
    )

    # ④ 时长 / 站内订单 / 优惠券（此时订单数 = 商品数量，与原脚本顺序一致）
    out["发货时长（小时）"] = (
        pd.to_datetime(out["发货时间"]) - pd.to_datetime(out["支付完成时间"])
    ).dt.total_seconds() / 3600
    out["揽件时长（小时）"] = (
        pd.to_datetime(out["揽件时间"]) - pd.to_datetime(out["支付完成时间"])
    ).dt.total_seconds() / 3600
    out["优惠券单量"] = np.where(out["平台优惠"].ne("-"), out["订单数"], 0)
    out["站内订单"] = np.where(
        out["APP渠道"].isin({"抖音", "抖音极速版", "抖音商城版"}), out["订单数"], 0
    )

    # ⑤ 销售金额：填充空值；订单数为 0 时归零
    out["销售金额"] = out[sales_col].fillna(0)
    out["订单数"] = out["总支付单量"] - out["售前退款单量"] - out["备货单量"]
    out["揽件订单数"] = out["总支付单量"] - out["揽件前退款单量"] - out["备货单量"]
    out.loc[out["订单数"] == 0, "销售金额"] = 0

    # ⑥ 状态指标（原 if/elif 链的等价向量化）
    done_after_sale = out["售后状态"].isin(DONE_AFTER_SALE)
    paid_done = out["订单状态"].eq("已完成") & done_after_sale
    refunded = out["售后状态"].eq("退款成功")
    shipped = out["订单状态"].eq("已发货") & done_after_sale

    out["在线支付已完成"] = np.where(paid_done, out["订单数"], 0)
    out["退回成功"] = np.where(refunded, out["订单数"], 0)
    out["已发货"] = np.where(shipped, out["订单数"], 0)
    out["售后中"] = np.where(
        ~(paid_done | refunded | shipped) & ~out["售后状态"].isin(AFTER_SALE_IN_PROGRESS_EXCLUDE),
        out["订单数"],
        0,
    )

    # ⑦ 复购 / 广告
    out["复购"] = np.where(out["广告渠道"].eq("无"), out["订单数"], 0)
    out["广告"] = np.where(out["广告渠道"].isin({"短视频", "商品卡"}), out["订单数"], 0)
    return out

## 9. 报表生成（参数化，两个口径共用）

In [10]:
# 广告消耗表（日期 × 产品名称）
df_consume = pd.read_excel(CONSUME_XLSX)[["日期", "产品名称", "消耗", "转化数"]]
_tmp = df_consume["日期"].str.extract(r"(\d+).*?(\d+).*?(\d+)")
df_consume["订单提交时间"] = _tmp[0] + "-" + _tmp[1] + "-" + _tmp[2]
df_consume = df_consume.drop(columns=["日期"])
# 与原脚本一致：pivot_table + np.sum（求和顺序不同会带来浮点尾差，这里保持原口径）
df_consume = pd.pivot_table(
    df_consume, index=["订单提交时间", "产品名称"], values=["消耗", "转化数"], aggfunc=np.sum
).reset_index()
df_consume["订单提交时间"] = pd.to_datetime(
    df_consume["订单提交时间"], format="%Y-%m-%d"
).dt.strftime("%Y-%m-%d")


def _pivot(df_in, value_cols):
    return pd.pivot_table(
        df_in, index=["订单提交时间", "产品名称"], values=value_cols, aggfunc=np.sum
    ).reset_index()


def _add_rates(frame):
    """计算转化 / 签收 / 退款等比率（未格式化）。"""
    frame["真实发货率"] = frame["订单数"] / frame["转化数"]
    frame["签收率"] = frame["在线支付已完成"] / frame["转化数"]
    frame["退回率"] = frame["退回成功"] / frame["转化数"]
    frame["转化成本"] = frame["消耗"] / frame["转化数"]
    frame["平均客单"] = frame["销售金额"] / frame["订单数"]
    frame["复购率"] = frame["复购"] / frame["转化数"]
    frame["对标支付签收率"] = frame["在线支付已完成"] / frame["总支付单量"]
    frame["发货时长"] = frame["发货时长（小时）"] / frame["订单数"]
    frame["揽件时长"] = frame["揽件时长（小时）"] / frame["揽件订单数"]
    frame["站内订单占比"] = frame["站内订单"] / frame["总支付单量"]
    frame["优惠订单占比"] = frame["优惠券单量"] / frame["总支付单量"]
    frame["对标揽件单量签收率"] = frame["在线支付已完成"] / frame["揽件订单数"]
    frame["对标揽件单量退回率"] = frame["退回成功"] / frame["揽件订单数"]
    frame["对标揽件单量复购率"] = frame["复购"] / frame["揽件订单数"]
    frame["售前退款率"] = frame["售前退款单量"] / frame["总支付单量"]
    frame["揽件前退款率"] = frame["揽件前退款单量"] / frame["总支付单量"]
    return frame


def _format_rates(frame):
    for col in RATE_COLUMNS:
        if col in frame.columns:
            frame[col] = frame[col].apply(lambda x: format(x, ".2%"))
    frame["转化成本"] = frame["转化成本"].round(decimals=2)
    frame["平均客单"] = frame["平均客单"].round(decimals=2)
    return frame


def build_tables(df_in, sales_col):
    """生成（汇总, 广告）两张表。"""
    df_metrics = add_metrics(df_in, sales_col=sales_col)

    # ---- 汇总表 ----
    final = _pivot(df_metrics, PIVOT_VALUES)[RESULT_COLUMNS]
    final = final.merge(df_consume, on=["订单提交时间", "产品名称"], how="left")
    final = _format_rates(_add_rates(final))
    final = final.rename(columns=RENAME_COLUMNS)
    final = final[FINAL_COLUMNS]

    # ---- 广告表（只看短视频 / 商品卡流量） ----
    df_advert = df_metrics[df_metrics["流量体裁"].isin({"短视频", "商品卡"})]
    final_advert = _pivot(df_advert, PIVOT_VALUES)
    final_advert = final_advert.merge(df_consume, on=["订单提交时间", "产品名称"], how="left")
    final_advert = _format_rates(_add_rates(final_advert))
    final_advert = final_advert.rename(columns=RENAME_COLUMNS)
    final_advert = final_advert[ADVERT_COLUMNS]

    return final, final_advert

## 10. 生成「直播汇总.xlsx」（全量口径）

In [11]:
final, final_advert = build_tables(df3, sales_col="销售金额")

with pd.ExcelWriter(SUMMARY_FILE, engine="openpyxl") as writer:
    final.to_excel(writer, sheet_name="汇总", index=False)
    final_advert.to_excel(writer, sheet_name="广告", index=False)
print(f"已导出 {SUMMARY_FILE}：汇总 {final.shape[0]} 行，广告 {final_advert.shape[0]} 行")

已导出 直播汇总.xlsx：汇总 21 行，广告 21 行


## 11. 生成「直播汇总-到货.xlsx」（到货 / 已签收口径）

In [12]:
final_arr, final_advert_arr = build_tables(df3_arrival, sales_col="到货销售金额")

with pd.ExcelWriter(SUMMARY_ARRIVAL_FILE, engine="openpyxl") as writer:
    final_arr.to_excel(writer, sheet_name="汇总", index=False)
    final_advert_arr.to_excel(writer, sheet_name="广告", index=False)
print(f"已导出 {SUMMARY_ARRIVAL_FILE}：汇总 {final_arr.shape[0]} 行，广告 {final_advert_arr.shape[0]} 行")

已导出 直播汇总-到货.xlsx：汇总 21 行，广告 21 行


## 12. 结果预览

In [13]:
final.head()

,日期,产品名称,广告单量,总发出单量,真实发货率,总支付单量,售前退款率,揽件前退款率,在线支付已完成,已发货,售后中,退回成功,签收率,退回率,复购率,消耗,发货时长,揽件时长,优惠订单占比,站内订单占比,对标支付签收率,对标揽件单量签收率,对标揽件单量退回率,对标揽件单量复购率,平均客单,客单,复购,备货单量,售前退款单量,揽件前退款单量,发货时长（小时）,揽件时长（小时）,站内订单,优惠券单量,揽件订单数
0,2026-10-16,2602#塑形收腹裤,326.0,331,101.53%,345,4.06%,9.28%,279,0,0,52,85.58%,15.95%,101.53%,8558.56,0.121107,8.693528,38.26%,80.29%,80.87%,89.14%,16.61%,105.75%,59.21,19596.9,331,0,14,32,40.086389,2721.074167,277,132,313
1,2026-10-17,2602#塑形收腹裤,283.0,269,95.05%,285,5.61%,11.58%,221,0,0,48,78.09%,16.96%,95.05%,7318.61,0.145398,7.332342,36.49%,74.04%,77.54%,87.70%,19.05%,106.75%,58.90,15843.1,269,0,16,33,39.111944,1847.750278,211,104,252
2,2026-10-18,2602#塑形收腹裤,174.0,172,98.85%,183,6.01%,8.20%,154,0,0,18,88.51%,10.34%,98.85%,4379.66,0.123936,7.511288,45.36%,72.13%,84.15%,91.67%,10.71%,102.38%,58.85,10122.8,172,0,11,15,21.316944,1261.896389,132,83,168
3,2026-10-19,2602#塑形收腹裤,200.0,191,95.50%,204,6.37%,11.76%,160,0,1,30,80.00%,15.00%,95.50%,4780.70,0.146278,9.774457,50.49%,60.29%,78.43%,88.89%,16.67%,106.11%,57.88,11055.9,191,0,13,24,27.939167,1759.402222,123,103,180
4,2026-10-20,2602#塑形收腹裤,225.0,218,96.89%,234,6.84%,11.11%,181,0,1,36,80.44%,16.00%,96.89%,5249.70,0.220455,7.709310,67.95%,67.09%,77.35%,87.02%,17.31%,104.81%,56.51,12318.2,218,0,16,26,48.059167,1603.536389,157,159,208


In [14]:
final_arr.head()

,日期,产品名称,广告单量,总发出单量,真实发货率,总支付单量,售前退款率,揽件前退款率,在线支付已完成,已发货,售后中,退回成功,签收率,退回率,复购率,消耗,发货时长,揽件时长,优惠订单占比,站内订单占比,对标支付签收率,对标揽件单量签收率,对标揽件单量退回率,对标揽件单量复购率,平均客单,客单,复购,备货单量,售前退款单量,揽件前退款单量,发货时长（小时）,揽件时长（小时）,站内订单,优惠券单量,揽件订单数
0,2026-10-16,2602#塑形收腹裤,326.0,307,94.17%,307,0.00%,0.00%,279,0,0,28,85.58%,8.59%,94.17%,8558.56,0.122173,8.652805,38.44%,79.48%,90.88%,90.88%,9.12%,100.00%,117.88,36189.3,307,0,0,0,37.507222,2656.411111,244,118,307
1,2026-10-17,2602#塑形收腹裤,283.0,242,85.51%,242,0.00%,0.00%,219,0,0,23,77.39%,8.13%,85.51%,7318.61,0.149384,7.335356,33.88%,72.73%,90.50%,90.50%,9.50%,100.00%,117.42,28415.8,242,0,0,0,36.150833,1775.156111,176,82,242
2,2026-10-18,2602#塑形收腹裤,174.0,164,94.25%,164,0.00%,0.00%,154,0,0,10,88.51%,5.75%,94.25%,4379.66,0.123359,7.519771,46.34%,72.56%,93.90%,93.90%,6.10%,100.00%,117.95,19343.6,164,0,0,0,20.230833,1233.242500,119,76,164
3,2026-10-19,2602#塑形收腹裤,200.0,176,88.00%,176,0.00%,0.00%,157,0,1,18,78.50%,9.00%,88.00%,4780.70,0.128127,9.969066,48.86%,59.09%,89.20%,89.20%,10.23%,100.00%,117.40,20662.4,176,0,0,0,22.550278,1754.555556,104,86,176
4,2026-10-20,2602#塑形收腹裤,225.0,204,90.67%,204,0.00%,0.00%,180,0,1,23,80.00%,10.22%,90.67%,5249.70,0.132168,7.673859,68.14%,65.20%,88.24%,88.24%,11.27%,100.00%,115.98,23659.6,204,0,0,0,26.962222,1565.467222,133,139,204
